In [1]:
import pandas as pd
import numpy as np
import os
import torch
import seaborn as sns


from scipy.stats import pearsonr, spearmanr
from matplotlib import pyplot as plt
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve, auc


from Bio import pairwise2
from Bio.Seq import Seq
from Bio.Align import substitution_matrices
from Bio import SeqIO

from random import sample



/home/labs/fleishman/itayta/.conda/envs/esm_env/lib/python3.10/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [ ]:
dataset_to_use = "gfp"

datasets_path = "./notebooks/%s/fitness_results/" % dataset_to_use
df = pd.read_csv(datasets[dataset_to_use])

normed_fitness_path = datasets_path + "normed_fitness_all.csv"
normed_fitness = pd.read_csv(normed_fitness_path)

fitness_path = datasets_path + "fitness_all.csv"
fitness = pd.read_csv(fitness_path)

active_indices = (df["inactive"] == False).to_numpy()
inactive_indices = (df["inactive"] == True).to_numpy()
active_df = df[active_indices]
inactive_df = df[inactive_indices]
active_nmuts = active_df["num_muts"].to_numpy()
active_fitness = fitness[active_indices]
active_normed_fitness = normed_fitness[active_indices]


In [ ]:
gfp_full_seq = (df[df["num_muts"] == 0]["full_seq"].item())

In [ ]:
gfp_full_seq

In [ ]:
pssm_file = open("pssm", "r")
pssm_lines_tmp = pssm_file.readlines()
pssm_lines = [[x for x in ln.split(" ") if x != ""] for ln in pssm_lines_tmp]
pssm_seq = "".join([ln[1] for ln in pssm_lines[3:]])
pssm = [[int(prob) for prob in ln[2:22]] for ln in pssm_lines[3:]]

assert pssm_seq == "KGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLYVQCFSRYPDHMKRHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKTRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITH"

pssm_vocab =pssm_lines[2][0:20]

pssm = np.array(pssm)
max = pssm.argmax(axis=1)
consensus = [pssm_vocab[i] for i in max]
consensus = "".join(consensus)

first_chunk = pssm_seq[:62]
second_chunk = pssm_seq[63:228]

pssm_first_chunk = pssm[0:62,:]
pssm_second_chunk = pssm[63:228,:]

first_chunk_consensus = consensus[:62]
second_chunk_consensus = consensus[63:228]
first_chunk_idx = gfp_full_seq.find(first_chunk)
second_chunk_idx = gfp_full_seq.find(second_chunk)

aligned_seq = "".join(["_"] * len(gfp_full_seq))
pssm_aligned_to_seq = aligned_seq[:first_chunk_idx] + first_chunk + aligned_seq[first_chunk_idx + len(first_chunk):second_chunk_idx] + second_chunk + aligned_seq[second_chunk_idx + len(second_chunk):]
consensus_aligned_seq = aligned_seq[:first_chunk_idx] + first_chunk_consensus + aligned_seq[first_chunk_idx + len(first_chunk):second_chunk_idx] + second_chunk_consensus + aligned_seq[second_chunk_idx + len(second_chunk):]


final_pssm = np.concatenate([np.zeros((len(range(0, first_chunk_idx)), pssm.shape[1])),
                             pssm_first_chunk,
                             np.zeros((len(range(first_chunk_idx+len(first_chunk), second_chunk_idx)), pssm.shape[1])),
                             pssm_second_chunk,
                             np.zeros((len(range(second_chunk_idx + len(second_chunk), len(gfp_full_seq))), pssm.shape[1])),])


blosum62 = substitution_matrices.load("BLOSUM62")


si = np.where(df.columns == "L42")[0][0]
ei = np.where(df.columns == "V224")[0][0]+1

muts_df = df.iloc[:,si:ei]
consensus_per_pos = [consensus_aligned_seq[int(pos[1:])-1] for pos in df.columns[si:ei].to_list()]
wt_per_pos =  [p[0] for p in df.columns[si:ei].to_list()]
mutated_pos = [int(pos[1:]) for pos in df.columns[si:ei].to_list()]


final_pssm_df = pd.DataFrame(final_pssm, columns=pssm_vocab)

for c, o in zip(consensus_per_pos, wt_per_pos):
    print(c, o)



In [ ]:

# Iterate through each row in muts_df (sequence variants)
# For each position/column, compute the BLOSUM62 score of the amino acid substitution vs consensus
blosum_scores_per_row = []
pssm_scores_per_row = []
for i, row in muts_df.iterrows():
    row_scores = []
    pssm_row_scores = []

    if i % 20000 == 0:
        print("Finished %d rows" % i)
    for j, (aa, consensus_aa) in enumerate(zip(row, consensus_per_pos)):
        try:
            score = blosum62[aa, consensus_aa]
        except Exception:
            # Fallback to 0 if invalid residue, e.g. nan or gap
            score = 0
        row_scores.append(score)
        pssm_row_scores.append(final_pssm_df.iloc[mutated_pos[j]-1][aa])
    blosum_scores_per_row.append(row_scores)
    pssm_scores_per_row.append(pssm_row_scores)

blosum_scores_per_row = np.array(blosum_scores_per_row)
pssm_scores_per_row = np.array(pssm_scores_per_row)

pssm_score = pssm_scores_per_row.sum(axis=1)
consensus_score = blosum_scores_per_row.sum(axis=1)

In [ ]:
blosum_scores_per_row[active_indices][168]

In [ ]:
example_pssm = pd.read_csv("/home/labs/fleishman/itayta/new_fitness_repo/fitness_learning/notebooks/notebooks/gfp/fitness_results/pssm_esm_8m.csv")
example_fitness = fitness["esm_8m"]

example_index = np.random.randint(0, df.shape[0])
example_row = df.iloc[example_index]
example_row_mutations = example_row[si:ei].to_list()

mutation_positions =[i for i,(x1,x2) in enumerate(zip(wt_per_pos, example_row_mutations)) if x1 != x2]
changes_per_positions = [(wt_per_pos[m], example_row_mutations[m]) for m in mutation_positions]
print(changes_per_positions)

sum_scores = []
for idx in range(len(mutation_positions)):

    m = mutation_positions[idx]
    print(wt_per_pos[m], example_row_mutations[m])
    wt = wt_per_pos[m]
    mt = example_row_mutations[m]
    score = np.log(example_pssm.iloc[m][mt]) - np.log(example_pssm.iloc[m][wt])
    print(score)
    sum_scores.append(score)

assert sum(sum_scores) - example_fitness.iloc[example_index] < 1e-05, "Sum of scores does not match fitness"




In [ ]:
print("######### Scores for #mutations == 0 idx (%d)" % np.where(df["num_muts"] == 0)[0][0])
print("PSSM score: ", pssm_score[np.where(df["num_muts"] == 0)[0][0]])
print("BLOSUM Consensus score: ", consensus_score[np.where(df["num_muts"] == 0)[0][0]])
print("Fitness: ", fitness.iloc[np.where(df["num_muts"] == 0)[0][0],: ])

In [ ]:
wt

In [ ]:
example_pssm.shape

In [ ]:

active_consensus_score = pssm_score[active_indices]

mut_range = range(2, 9)  # 2 to 8

for fit_col in active_fitness.columns:
    fig, axes = plt.subplots(2, len(mut_range), figsize=(4*len(mut_range), 8), sharey='row')
    spearman_raw = []
    spearman_norm = []
    col_active_fitness = active_fitness[fit_col].to_numpy()
    col_active_normed_fitness = active_normed_fitness[fit_col].to_numpy()
    for idx, nmut in enumerate(mut_range):
        mask = (active_nmuts == nmut)
        x = active_consensus_score[mask]
        
        # Row 0: Raw fitness correlation plots
        y_raw = col_active_fitness[mask]
        if len(x) > 0 and hasattr(y_raw, "shape"):
            y_raw = np.asarray(y_raw).reshape(-1)
        # Correlation
        if len(x) > 1 and len(y_raw) > 1:
            corr_raw, pval_raw = spearmanr(x, y_raw)
            spearman_raw.append(corr_raw)
            axes[0, idx].scatter(x, y_raw, alpha=0.5)
            axes[0, idx].set_title(f"{fit_col}\nnmut={nmut}\nSpearman r={corr_raw:.2f}")
        else:
            axes[0, idx].set_title(f"{fit_col}\nnmut={nmut}\nInsufficient data")
        axes[0, idx].set_xlabel("Consensus Score")
        axes[0, idx].set_ylabel("Raw Fitness")

        # Row 1: Normed fitness correlation plots
        y_norm = col_active_normed_fitness[mask]
        if len(x) > 0 and hasattr(y_norm, "shape"):
            y_norm = np.asarray(y_norm).reshape(-1)
        if len(x) > 1 and len(y_norm) > 1:
            corr_norm, pval_norm = spearmanr(x, y_norm)
            spearman_norm.append(corr_norm)
            axes[1, idx].scatter(x, y_norm, alpha=0.5)
            axes[1, idx].set_title(f"{fit_col}\nnmut={nmut}\nSpearman r={corr_norm:.2f}")
        else:
            axes[1, idx].set_title(f"{fit_col}\nnmut={nmut}\nInsufficient data")
        axes[1, idx].set_xlabel("Consensus Score")
        axes[1, idx].set_ylabel("Normed Fitness")

    plt.tight_layout()
    plt.show()


In [ ]:
spearmanr(active_fitness["esm_8m"], active_consensus_score)